<a href="https://colab.research.google.com/github/arxsyy/MachineLearning_2026/blob/main/JS09_Tugas2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install opendatasets faiss-cpu hnswlib annoy -q

In [4]:
import os
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [5]:
# 1. Set Token API Kaggle sesuai dengan yang ada di layarmu
os.environ['KAGGLE_API_TOKEN'] = "KGAT_bf5bebc2dde733876ff3a6b7233f3d7e"

# 2. Download dan ekstrak dataset langsung dari Kaggle
!kaggle datasets download -d bwandowando/spotify-songs-with-attributes-and-lyrics --unzip

# 3. Load dataset hasil download
file_path = 'songs_with_attributes_and_lyrics.csv'
df = pd.read_csv(file_path)

Dataset URL: https://www.kaggle.com/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics
License(s): CC-BY-NC-SA-4.0
100% 894M/894M [00:13<00:00, 70.9MB/s]



In [6]:
# 2. Pilih hanya fitur numerik dan lakukan normalisasi
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# Bersihkan data dari NaN agar algoritma ANN tidak error
df_clean = df.dropna(subset=features).reset_index(drop=True)
X = df_clean[features].values

# Standarisasi fitur menggunakan StandardScaler
scaler = StandardScaler()
# FAISS dan HNSWLIB membutuhkan tipe data float32
X_scaled = scaler.fit_transform(X).astype('float32')

# Konfigurasi pencarian
k = 10  # jumlah nearest neighbors
d = X_scaled.shape[1] # dimensi fitur

# Ambil satu lagu acak sebagai target pencarian (query)
query_idx = 0
query_vector = X_scaled[query_idx].reshape(1, -1)

print(f"Mencari {k} track terdekat untuk lagu pada indeks {query_idx}...\n")

Mencari 10 track terdekat untuk lagu pada indeks 0...



Pencarian dan Perbandingan Algoritma

In [7]:
# A. Exact Nearest Neighbors (Sklearn)
start_time = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
distances_nn, indices_nn = nn.kneighbors(query_vector)
time_nn = time.time() - start_time

# B. FAISS
start_time = time.time()
index_faiss = faiss.IndexFlatL2(d)
index_faiss.add(X_scaled)
distances_faiss, indices_faiss = index_faiss.search(query_vector, k)
time_faiss = time.time() - start_time

# C. ANNOY
start_time = time.time()
index_annoy = AnnoyIndex(d, 'euclidean')
for i in range(X_scaled.shape[0]):
    index_annoy.add_item(i, X_scaled[i])
index_annoy.build(10) # 10 trees
indices_annoy, distances_annoy = index_annoy.get_nns_by_vector(query_vector[0], k, include_distances=True)
time_annoy = time.time() - start_time

# D. HNSWLIB
start_time = time.time()
index_hnsw = hnswlib.Index(space='l2', dim=d)
# ef_construction & M menentukan akurasi vs waktu build
index_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
index_hnsw.add_items(X_scaled)
index_hnsw.set_ef(50) # ef pencarian
indices_hnsw, distances_hnsw = index_hnsw.knn_query(query_vector, k=k)
time_hnsw = time.time() - start_time


Output Hasil Perbandingan

In [8]:
print("=== HASIL INDEKS TRACK TERDEKAT ===")
print(f"Exact NN : {indices_nn[0]}")
print(f"FAISS    : {indices_faiss[0]}")
print(f"ANNOY    : {indices_annoy}")
print(f"HNSWLIB  : {indices_hnsw[0]}\n")

print("=== PERBANDINGAN WAKTU EKSEKUSI ===")
print(f"Exact NN : {time_nn:.6f} detik")
print(f"FAISS    : {time_faiss:.6f} detik")
print(f"ANNOY    : {time_annoy:.6f} detik")
print(f"HNSWLIB  : {time_hnsw:.6f} detik")

=== HASIL INDEKS TRACK TERDEKAT ===
Exact NN : [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]
FAISS    : [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]
ANNOY    : [0, 764272, 749223, 625227, 833164, 676733, 523698, 85956, 888081, 763625]
HNSWLIB  : [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]

=== PERBANDINGAN WAKTU EKSEKUSI ===
Exact NN : 0.233712 detik
FAISS    : 0.122823 detik
ANNOY    : 19.896371 detik
HNSWLIB  : 229.721011 detik
